# SegmentAE Example: Custom Autoencoder Models

This notebook demonstrates how to use SegmentAE with custom Keras/TensorFlow autoencoder models for anomaly detection.

## Features demonstrated:
- Custom autoencoder architecture using Keras Sequential API
- Multiple threshold ratio evaluation
- Comprehensive anomaly detection workflow

## Import Dependencies

In [ ]:
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.models import Sequential

from segmentae import Clustering, Preprocessing, SegmentAE
from segmentae.data_sources import load_dataset

## Data Loading

In [ ]:
train, test, target = load_dataset(
    dataset_selection="htru2_dataset",  # Options: 'default_credit_card', 'htru2_dataset', 'shuttle_148'
    split_ratio=0.75,
)

test, future_data = train_test_split(test, train_size=0.75, random_state=5)

# Resetting the index is optional: rows are handled by position and results
# keep the index of the data they were computed from
train = train.reset_index(drop=True)
test = test.reset_index(drop=True)
future_data = future_data.reset_index(drop=True)

X_train = train.drop(columns=[target]).copy()
X_test, y_test = test.drop(columns=[target]).copy(), test[target].astype(int)
X_future_data = future_data.drop(columns=[target]).copy()

print("Dataset: HTRU2 Pulsar Detection")
print(f"  Training samples: {len(X_train)}")
print(f"  Test samples: {len(X_test)}")
print(f"  Future samples: {len(X_future_data)}")
print(f"  Features: {X_train.shape[1]}")

## Data Preprocessing

In [ ]:
# MinMaxScaler maps the features to [0, 1], the range a sigmoid output layer can
# reconstruct. With "StandardScaler" or "RobustScaler", use a linear output layer instead.
pr = Preprocessing(
    encoder="IFrequencyEncoder",  # "LabelEncoder", "OneHotEncoder" || #EncoderType.IFREQUENCY,
    scaler="MinMaxScaler",  # "StandardScaler", "RobustScaler"  || #ScalerType.MINMAX,
    imputer=None,  # Optional Imputation
)

pr.fit(X=X_train)
X_train = pr.transform(X=X_train)
X_test = pr.transform(X=X_test)
X_future_data = pr.transform(X=X_future_data)

print("✓ Preprocessing completed")
print(f"  Final shape: {X_train.shape}")

## Custom Autoencoder Architecture

Build a custom autoencoder using Keras Sequential API with encoder-decoder structure.

In [ ]:
print("Num GPUs Available: ", len(tf.config.list_physical_devices("GPU")))
print(f"Input shape: {X_train.shape[1]}")

# Define the encoder
encoder = Sequential(
    [
        Input(shape=(X_train.shape[1],)),
        Dense(32, activation="relu"),
        Dense(24, activation="relu"),
        Dense(16, activation="relu"),
        Dense(8, activation="relu"),
    ]
)

print("✓ Encoder architecture defined")
print(f"  Encoder layers: {len(encoder.layers)}")

In [ ]:
# Define the decoder
decoder = Sequential(
    [
        Input(shape=(8,)),
        Dense(16, activation="relu"),
        Dense(24, activation="relu"),
        Dense(32, activation="relu"),
        Dense(X_train.shape[1], activation="sigmoid"),  # reconstructs MinMax-scaled values
    ]
)

print("✓ Decoder architecture defined")
print(f"  Decoder layers: {len(decoder.layers)}")

In [ ]:
# Combine encoder and decoder into an autoencoder model
autoencoder = Sequential([encoder, decoder])

# Compile the model
autoencoder.compile(optimizer="adam", loss="mse")

print("✓ Complete autoencoder model created")
print(f"  Total parameters: {autoencoder.count_params():,}")

# Display model architecture
autoencoder.summary()

## Train the Autoencoder

In [ ]:
# Train the model
print("Training custom autoencoder...")
history = autoencoder.fit(
    X_train, 
    X_train, 
    epochs=50, 
    batch_size=None, 
    validation_split=0.1,
    verbose=1
)

print("✓ Autoencoder training completed")

## Training History Visualization

In [ ]:
import matplotlib.pyplot as plt

# Plot training history
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

print(f"Final training loss: {history.history['loss'][-1]:.6f}")
print(f"Final validation loss: {history.history['val_loss'][-1]:.6f}")

## Clustering Implementation

In [ ]:
cl_model = Clustering(
    cluster_model=["KMeans"],  # "KMeans", "MiniBatchKMeans", "GMM"
    n_clusters=3,
)

cl_model.clustering_fit(X=X_train)
print("✓ Clustering completed with KMeans")

## SegmentAE Integration

In [ ]:
# Any object with a predict(X) method can be used as the autoencoder
sg = SegmentAE(ae_model=autoencoder, cl_model=cl_model)

# Train Reconstruction
sg.reconstruction(
    input_data=X_train,
    threshold_metric="mse",  # || "mse", "mae", "rmse", "max_error"
)

print("✓ SegmentAE reconstruction phase completed")
print("  Baseline threshold per cluster (mean training reconstruction error):")
for cluster_id, threshold in sg.thresholds.items():
    print(f"    Cluster {cluster_id}: {threshold:.6f}")

## Single Threshold Evaluation

In [ ]:
results = sg.evaluation(input_data=X_test, target_col=y_test, threshold_ratio=2.0)

# Access metadata by cluster
preds_test, recon_metrics_test = sg.preds_test, sg.reconstruction_test

print("Evaluation Results (Threshold Ratio = 2.0):")
print(results['global metrics'])

## Multiple Threshold Ratio Evaluation

In [ ]:
threshold_ratios = [0.75, 1, 1.5, 2, 3, 4]

global_results = pd.concat(
    [
        sg.evaluation(input_data=X_test, target_col=y_test, threshold_ratio=thr)["global metrics"]
        for thr in threshold_ratios
    ]
)

print("\nPerformance across different thresholds:")
print(global_results[['Threshold Ratio', 'Accuracy', 'F1 Score', 'Precision', 'Recall']])

## Final Anomaly Detection

Apply the best threshold configuration for final predictions.

In [ ]:
best_ratio = global_results.sort_values(by="Accuracy", ascending=False).iloc[0]["Threshold Ratio"]
print(f" Best threshold ratio: {best_ratio}")

predictions = sg.detections(input_data=X_future_data, threshold_ratio=best_ratio)

n_anomalies = predictions['Predicted Anomalies'].sum()
total_samples = len(predictions)
anomaly_rate = n_anomalies / total_samples

print("\n Detection Results:")
print(f"  Detected {n_anomalies} anomalies out of {total_samples} samples")
print(f"  Anomaly rate: {anomaly_rate:.2%}")

# Show cluster-wise breakdown
cluster_breakdown = predictions.groupby('Cluster')['Predicted Anomalies'].agg(['count', 'sum', 'mean'])
cluster_breakdown.columns = ['Total Samples', 'Anomalies', 'Anomaly Rate']
print("\n Cluster-wise Breakdown:")
print(cluster_breakdown)